# 04 — LTV Prediction

Leakage-aware LTV regression and model diagnostics.

**Project:** Marketing Analytics Causal & LTV Lab  
**Phase:** Phase 1 — Customer Analytics, Retention, Churn and LTV Baseline

> This notebook is designed as a hands-on learning notebook. Run each section, inspect the output, and discuss the interpretation before moving to the next step.


## 1. Notebook objective

This notebook predicts `LTV` and compares two modeling views:

1. **Snapshot model:** uses most available customer information.
2. **Leakage-aware model:** removes variables that may directly encode lifetime value.

This is important because LTV datasets often contain accumulated post-outcome variables.


In [ ]:
# Core imports
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix
)
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_RAW = Path("../data/raw/digital_wallet_ltv_dataset.csv")
DATA_PROCESSED = Path("../data/processed")
REPORTS = Path("../reports")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PROCESSED / 'wallet_churn_features.csv') if (DATA_PROCESSED / 'wallet_churn_features.csv').exists() else pd.read_csv(DATA_RAW)
df.head()


## 2. Helper functions


In [ ]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

def evaluate_regression(y_true, y_pred):
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": rmse(y_true, y_pred),
        "r2": r2_score(y_true, y_pred)
    }

def build_preprocess(X):
    num = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
    cat = X.select_dtypes(include=["object", "category"]).columns.tolist()

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", numeric_pipe, num),
        ("cat", categorical_pipe, cat)
    ]), num, cat


## 3. Define feature sets


In [ ]:
TARGET = "LTV"

base_drop = ["Customer_ID", TARGET]

snapshot_features = [c for c in df.columns if c not in base_drop]

leakage_candidates = [
    "Total_Spent",
    "Loyalty_Points_Earned",
    "Cashback_Received",
    "churn_risk_score",
    "risk_decile"
]

leakage_aware_features = [
    c for c in snapshot_features 
    if c not in leakage_candidates
]

print("Snapshot feature count:", len(snapshot_features))
print("Leakage-aware feature count:", len(leakage_aware_features))
print("Removed possible leakage:", [c for c in leakage_candidates if c in df.columns])


## 4. Model training function


In [ ]:
def run_ltv_experiment(df, features, experiment_name):
    X = df[features]
    y = df[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=RANDOM_STATE
    )

    preprocess, num_features, cat_features = build_preprocess(X)

    models = {
        "mean_baseline": None,
        "linear_regression": LinearRegression(),
        "ridge": Ridge(alpha=10.0),
        "random_forest": RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            random_state=RANDOM_STATE
        ),
        "gradient_boosting": GradientBoostingRegressor(random_state=RANDOM_STATE)
    }

    rows = []
    fitted = {}

    for name, model in models.items():
        if name == "mean_baseline":
            pred = np.repeat(y_train.mean(), len(y_test))
            fitted[name] = None
        else:
            pipe = Pipeline([
                ("preprocess", preprocess),
                ("model", model)
            ])
            pipe.fit(X_train, y_train)
            pred = pipe.predict(X_test)
            fitted[name] = pipe

        metrics = evaluate_regression(y_test, pred)
        metrics["model"] = name
        metrics["experiment"] = experiment_name
        rows.append(metrics)

    results = pd.DataFrame(rows).sort_values("rmse")
    return results, fitted, (X_train, X_test, y_train, y_test)

snapshot_results, snapshot_models, snapshot_split = run_ltv_experiment(df, snapshot_features, "snapshot")
leakage_results, leakage_models, leakage_split = run_ltv_experiment(df, leakage_aware_features, "leakage_aware")

all_results = pd.concat([snapshot_results, leakage_results], ignore_index=True)
display(all_results.sort_values(["experiment", "rmse"]))


## 5. Select production-style model


In [ ]:
# Prefer leakage-aware unless snapshot model is intentionally used for descriptive scoring.
production_results = leakage_results.sort_values("rmse")
best_model_name = production_results.iloc[0]["model"]

print("Selected model:", best_model_name)
print(production_results.iloc[0])

best_pipe = leakage_models[best_model_name]

X_all = df[leakage_aware_features]
df_pred = df.copy()
df_pred["predicted_ltv"] = best_pipe.predict(X_all)

display(df_pred[["Customer_ID", "LTV", "predicted_ltv"]].head())


## 6. Prediction diagnostics


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(df_pred["LTV"], df_pred["predicted_ltv"], alpha=0.4)
plt.xlabel("Actual LTV")
plt.ylabel("Predicted LTV")
plt.title("Actual vs Predicted LTV")
plt.show()

df_pred["ltv_error"] = df_pred["predicted_ltv"] - df_pred["LTV"]

plt.figure(figsize=(8, 4))
df_pred["ltv_error"].hist(bins=40)
plt.title("LTV Prediction Error")
plt.xlabel("Prediction error")
plt.ylabel("Customer count")
plt.show()


## 7. Feature importance using permutation importance


In [ ]:
X_train, X_test, y_train, y_test = leakage_split

if best_pipe is not None:
    perm = permutation_importance(
        best_pipe, X_test, y_test,
        n_repeats=5,
        random_state=RANDOM_STATE,
        scoring="neg_root_mean_squared_error"
    )

    importance = pd.DataFrame({
        "feature": X_test.columns,
        "importance": perm.importances_mean
    }).sort_values("importance", ascending=False)

    display(importance.head(15))

    plt.figure(figsize=(8, 5))
    importance.head(12).sort_values("importance").plot(kind="barh", x="feature", y="importance", legend=False)
    plt.title("Permutation Importance - Leakage-aware LTV Model")
    plt.show()


## 8. Save predictions


In [ ]:
output = DATA_PROCESSED / "wallet_ltv_predictions.csv"
df_pred.to_csv(output, index=False)
print(f"Saved: {output}")


## Discussion prompts

1. Why can the snapshot model be misleading?
2. Which features may cause target leakage?
3. Why might a simpler model be better for business explanation?
4. How would you explain LTV prediction error to a marketing manager?
